# Agent to Agent(A2A) Multi-Agent 협업 프로토콜
Agent to Agent (A2A) 프로토콜은 서로 다른 목적이나 능력을 가진 인공지능 에이전트들이 사람의 개입 없이도 서로 통신하고, 협업하며, 복잡한 과제를 해결하기 위해 정의된 표준 규약을 의미합니다.
쉽게 말해, "에이전트들끼리 대화하는 공용 언어와 업무 규칙"이라고 이해하시면 됩니다.

1. A2A 프로토콜의 핵심 개념
과거에는 하나의 거대한 모델(Monolithic LLM)이 모든 일을 처리했다면, 이제는 특화된 작은 에이전트들이 팀을 이루어 일하는 방식이 주류가 되고 있습니다. 이때 에이전트 간의 '악수(Handshake)'와 '업무 전달'을 가능하게 하는 것이 A2A 프로토콜입니다.

2. 주요 동작 메커니즘
A2A 협업은 보통 다음과 같은 단계로 이루어집니다.

- 발견 (Discovery): 메인 에이전트가 특정 작업을 수행할 수 있는 다른 전문 에이전트(예: 코딩 에이전트, 검색 에이전트)가 누구인지 식별합니다.
- 협상 및 할당 (Negotiation): 작업을 수행할 에이전트에게 필요한 데이터와 권한을 전달하고, 수행 가능 여부를 확인합니다.
- 데이터 교환 (Data Exchange): JSON-RPC나 REST API 등을 통해 정형화된 메시지 규격으로 정보를 주고받습니다. (앞서 질문하신 message/send 형식이 그 예입니다.)
- 통합 (Integration): 각 에이전트가 보내온 부분적인 결과물들을 취합하여 최종 사용자에게 전달합니다.

3. 최근의 흐름: MCP (Model Context Protocol)
최근 이 분야에서 가장 주목받는 표준 중 하나는 Anthropic이 발표한 MCP(Model Context Protocol)입니다.
- 목적: 데이터 소스(Google Drive, Slack, GitHub 등)와 AI 모델 간의 연결을 표준화.
- 특징: 에이전트가 외부 도구를 사용할 때 제각각인 API 호출 방식을 하나로 통일하여, 한 번 구현한 도구를 여러 에이전트가 즉시 공유할 수 있게 합니다.

###  A2A Agent 예제 사용법 

1. 아래 %%writefile이 들어 있는 셀을 실행시켜 파이썬 소스를 생성 해놓는다
2. 최상위 폴더에 .env 파일을 만들고 아래 내용을 설정해 넣는다(프로젝트에 자신의 프로젝트 ID를 입력하여 사용한다)
   
GOOGLE_GENAI_USE_VERTEXAI=TRUE

GOOGLE_CLOUD_PROJECT=byounghwa-go

GOOGLE_CLOUD_LOCATION=global

4. Terminal을 실행 시킨 다음 아래 명령을 실행한다
   
python a2a_agent.py

5. "+" 버튼을 눌러 새 Terminal을 실행 시킨 다음 curl_a2a_agent.txt 명령 스크립트를 복사하여 붙여 넣어 실행시킨다

In [4]:
!pip install a2a-sdk

In [11]:
%%writefile a2a_agent.py
# 간단한 인사 에이전트 개발 
import uvicorn

from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import (
    AgentCard, AgentCapabilities, AgentSkill, TaskState, Part, TextPart
)
from a2a.utils import new_agent_text_message, new_task

# Step 1: A2A 로직 정의 - 에이전트의 핵심 동작 설계
# A2A 서버의 요청을 받아 에이전트의 핵심 로직을 실행하는 클래스
class SimpleGreetingExecutor(AgentExecutor):
    """간단한 인사말을 반환하는 A2A AgentExecutor 구현체."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        """클라이언트 요청 시, 정해진 인사말을 반환합니다."""
        # A2A 프로토콜의 작업(Task) 상태를 관리하는 객체
        task = context.current_task or new_task(context.message)
        updater = TaskUpdater(event_queue, task.id, task.context_id)

        # 클라이언트에게 작업이 진행 중임을 알림
        await updater.update_status(TaskState.working, message=new_agent_text_message("..."))

        # 최종 결과 메시지를 'Artifact' 형태로 클라이언트에게 전송
        final_message = "안녕하세요! 간단한 A2A 에이전트입니다."
        await updater.add_artifact([Part(root=TextPart(text=final_message))])
       
        # 클라이언트에게 작업이 완료되었음을 알림
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass

# Step 2: 서버 실행 - 에이전트를 외부에 노출
def main():
    """A2A 에이전트의 명함(AgentCard)을 정의하고 서버를 실행합니다."""
   
    # 2-1. 에이전트의 기능(Skill) 정의
    greeting_skill = AgentSkill(
        id="greeting-skill",
        name="Greeting Skill",
        description="간단한 인사말을 제공하는 스킬입니다.",
        tags=["greeting", "example"],
        examples=["인사해줘"],
    )

    # 2-2. AgentCard 생성
    agent_card = AgentCard(
        name="Greeting Agent",
        description="A2A 프로토콜을 사용한 간단한 인사 에이전트",
        url="http://localhost:8000",
        version="1.0.0",
        skills=[greeting_skill],
        capabilities=AgentCapabilities(streaming=True),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
    )

    # 2-3. A2A 서버 구성
    executor = SimpleGreetingExecutor()
    request_handler = DefaultRequestHandler(
        agent_executor=executor,
        task_store=InMemoryTaskStore(),
    )
    server_app = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler,
    )
    app = server_app.build()

    # 2-4. Uvicorn을 사용하여 A2A 서버 실행
    print("🚀 간단한 인사 A2A 에이전트 서버를 시작합니다.")
    uvicorn.run(app, host="0.0.0.0", port=8000)

if __name__ == "__main__":
    main()

Overwriting a2a_agent.py


### LangChain의 LangGraph를 연동한 A2A 에이전트

####  LangGraph A2A Agent 예제 사용법 
1. 아래 %%writefile이 들어 있는 셀을 실행시켜 파이썬 소스를 생성 해놓는다
2. 최상위 폴더에 .env 파일을 만들고 아래 내용을 설정해 넣는다(프로젝트에 자신의 프로젝트 ID를 입력하여 사용한다)
   
GOOGLE_GENAI_USE_VERTEXAI=TRUE

GOOGLE_CLOUD_PROJECT=byounghwa-go

GOOGLE_CLOUD_LOCATION=global

4. Terminal을 실행 시킨 다음 아래 명령을 실행한다
   
python langraph_a2a_agent.py

5. "+" 버튼을 눌러 새 Terminal을 실행 시킨 curl_langraph_a2a_agent.txt 명령 스크립트를 복사하여 붙여 넣어 실행시킨다

In [18]:
# !pip install langchain_google_vertexai
# !pip install langgraph

In [13]:
%%writefile langraph_a2a_agent.py

# LangGraph를 연동한 Q&A 에이전트 개발
import os
import uvicorn
import asyncio
from typing import TypedDict, Annotated, Sequence, Any

# A2A 및 LangChain 관련 라이브러리 임포트
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import AgentCard, AgentCapabilities, AgentSkill, TaskState, Part, TextPart
from a2a.utils import new_agent_text_message, new_task

from langchain_core.messages import BaseMessage, AIMessage
from langchain_google_vertexai import ChatVertexAI
from langgraph.graph import StateGraph, END

GOOGLE_GENAI_USE_VERTEXAI = os.getenv("GOOGLE_GENAI_USE_VERTEXAI")
GOOGLE_CLOUD_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")
GOOGLE_CLOUD_LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION")

# Step 1: LangGraph 로직 정의 - 에이전트 설계
# LangGraph의 대화 흐름(상태)을 관리하는 객체 정의
def add_messages(previous_conversation: list, conversation: list) -> list:
    """대화 기록을 누적하기 위한 간단한 함수"""
    return previous_conversation + conversation

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# LangGraph 기반의 에이전트를 생성하는 함수
def create_langgraph_app():
    """간단한 질의응답 LangGraph 워크플로우를 생성하고 컴파일합니다."""
   
    # 1-1. LLM 모델 설정
    model = ChatVertexAI(model_name="gemini-2.5-flash", temperature=0)

    # 1-2. Graph의 노드(Node)에서 수행할 함수 정의
    # call_model: LLM을 호출하여 사용자의 질문에 답변
    def call_model(state: AgentState):
        response = model.invoke(state["messages"])
        return {"messages": [response]}

    # 1-3. Graph 워크플로우 구성
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", call_model)
    workflow.set_entry_point("agent")
    workflow.add_edge("agent", END)
   
    return workflow.compile()

# Step 2: A2A 프로토콜과 LangGraph 연결 - 통신 채널 구축
# A2A 서버의 요청을 받아 LangGraph 에이전트를 실행시키는 클래스
class LangGraphA2AExecutor(AgentExecutor):
    def __init__(self, graph: Any):
        self.graph = graph

    # LangGraph 스트리밍 응답에서 텍스트만 추출하는 헬퍼 함수
    def _extract_text_from_chunk(self, chunk: Any) -> str:
        if not chunk:
            return ""
       
        messages = chunk.get("agent", {}).get("messages", []) or chunk.get("messages", [])
       
        if messages and isinstance(messages[-1], AIMessage):
            return messages[-1].content
        return ""

    # A2A 클라이언트의 요청을 실제로 처리하는 메인 함수
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_query = context.get_user_input()
       
        task = context.current_task or new_task(context.message)
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.start_work()

        config = {"configurable": {"thread_id": task.id}}
        graph_input = {"messages": [("human", user_query)]}
       
        final_result_chunk = None
        accumulated_text = ""

        async for chunk in self.graph.astream(graph_input, config=config):
            final_result_chunk = list(chunk.values())[0]
            partial_text = self._extract_text_from_chunk(final_result_chunk)
           
            if partial_text and partial_text != accumulated_text:
                new_delta = partial_text[len(accumulated_text):]
                await updater.update_status(
                    TaskState.working,
                    message=new_agent_text_message(new_delta, task.context_id, task.id)
                )
                accumulated_text = partial_text
       
        final_text = self._extract_text_from_chunk(final_result_chunk)
        await updater.add_artifact([Part(root=TextPart(text=final_text))])
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        """이 예제에서는 작업 취소를 지원하지 않습니다."""
        pass

# Step 3: 서버 실행 - 에이전트를 외부에 노출
async def main():
    """A2A 서버 구성 및 실행을 담당합니다."""

    # 3-1. LangGraph 앱 생성
    graph_app = create_langgraph_app()

    # 3-2. AgentCard 정의
    agent_card = AgentCard(
        name="LangGraph Q&A Agent",
        description="LangGraph 기반의 간단한 A2A 질의응답 에이전트",
        url="http://localhost:8000",
        version="1.0.0",
        skills=[
            AgentSkill(
                id="langgraph-qa-skill",
                name="LangGraph Q&A",
                description="LangGraph와 Gemini를 사용하여 일반적인 질문에 답변합니다.",
                tags=["qa", "langgraph", "llm"],
                examples=["LangChain과 LangGraph의 차이점은 무엇인가요?"],
            )
        ],
        capabilities=AgentCapabilities(streaming=True),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
    )

    # 3-3. A2A 서버 구성
    executor = LangGraphA2AExecutor(graph=graph_app)
    request_handler = DefaultRequestHandler(
        agent_executor=executor,
        task_store=InMemoryTaskStore(),
    )
    server_app = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler,
    )
    app = server_app.build()

    # 3-4. Uvicorn을 사용하여 A2A 서버 실행
    print("🚀 간단한 LangGraph A2A 에이전트 서버를 시작합니다.")
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()

if __name__ == "__main__":
    asyncio.run(main())

Overwriting langraph_a2a_agent.py


In [20]:
# 출력 필터링 명령 설명
# sed 's/^data: //'	접두사 제거	서버 응답이 data: {"jsonrpc": ...} 형식으로 올 때, 앞의 data: 문자열을 지워 순수한 JSON 형태만 남깁니다.
# grep '"kind":"artifact-update"'	특정 필터링	전체 로그 중 kind 값이 artifact-update인 줄만 골라냅니다. (코드 블록이나 문서 생성 같은 핵심 데이터만 추출)
# jq	JSON 정렬/출력	한 줄로 길게 늘어진 JSON 데이터를 들여쓰기(Pretty-print)하여 사람이 읽기 편하게 터미널에 뿌려줍니다.